In [15]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

import warnings
warnings.filterwarnings('ignore')
# Load the training dataset
file_path = "Phase 1 Predictions Output_capped.xlsx"

# Load all sheets from the Excel file
xls = pd.ExcelFile(file_path)
xls_t = pd.ExcelFile("Phase 1 Training Dataset.xlsx")
sheet_names = xls.sheet_names
# Read all sheets into a dictionary of dataframes
nh_data = {sheet: xls.parse(sheet) for sheet in sheet_names}

nh_data_2 = {sheet: xls_t.parse(sheet) for sheet in sheet_names}

In [17]:
# Process each group (sheet) in the dictionary
for group in nh_data.keys():
    # Get the DataFrame for this group
    df = nh_data[group]
    
    # Set column names directly to match the desired format
    # This replaces any existing column names with our new ones
    df.columns = [
        'Date',
        'CNA Point Prediction', 'CNA Lower Bound', 'CNA Upper Bound',
        'LPN Point Prediction', 'LPN Lower Bound', 'LPN Upper Bound',
        'RN Point Prediction', 'RN Lower Bound', 'RN Upper Bound'
    ]
    
    # Format date column if needed
    try:
        # Convert to datetime and then format as YYYY-MM-DD
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.dropna(subset=['Date'])  # Remove rows with invalid dates
        df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    except Exception as e:
        print(f"Error processing dates in group {group}: {e}")
        continue
    
    # Reset the index for clean display
    df = df.reset_index(drop=True)
    
    # Update the dictionary with the processed DataFrame
    nh_data[group] = df

# Display the first group to check the results
print(f"Processed group 'Group 1' with shape: {nh_data['Group 1'].shape}")
display(nh_data['Group 1'].head())

Processed group 'Group 1' with shape: (91, 10)


,Date,CNA Point Prediction,CNA Lower Bound,CNA Upper Bound,LPN Point Prediction,LPN Lower Bound,LPN Upper Bound,RN Point Prediction,RN Lower Bound,RN Upper Bound
0,2024-04-01,131.306229,104.443649,172.961166,67.731453,49.326162,95.129028,26.875391,18.150148,39.092628
1,2024-04-02,147.210159,108.637896,178.106359,73.200439,51.302617,98.031614,28.971222,18.87531,40.654516
2,2024-04-03,156.035187,115.154645,186.962555,78.93647,53.036331,101.340691,31.270206,18.655682,40.18147
3,2024-04-04,157.37323,115.994509,187.395533,75.161659,52.781322,100.848887,25.861721,18.964732,40.847117
4,2024-04-05,150.569626,112.347905,182.214355,72.107544,48.267686,93.053167,30.397608,18.564472,39.985017


In [18]:
"""Load training data from Excel file with multiple sheets"""
file_path = "./Phase 1 Training Dataset.xlsx"
xl = pd.ExcelFile(file_path)
groups = xl.sheet_names
train_df = pd.DataFrame()
test_df = pd.DataFrame()
for sheet in groups:
    df = pd.read_excel(file_path, sheet_name=sheet)
    
    # Skip the first row which contains staff type labels
    df = df.iloc[1:]
    
    # Convert the date column to datetime
    df['Date'] = pd.to_datetime(df.iloc[:, 0])
    
    # Sort by date to ensure temporal order
    df = df.sort_values('Date')
    
    # Calculate split point (80% train, 20% test)
    split_idx = int(len(df) * 0.8)
    
    for i in range(5):  # 5 NHs per group
        start_col = i * 4  # Each NH has 4 columns (NH No., CNA, LPN, RN)
        if start_col + 3 >= len(df.columns):
            break
            
        # Extract data for each staff type
        cna_data = pd.to_numeric(df.iloc[:, start_col + 1], errors='coerce')
        lpn_data = pd.to_numeric(df.iloc[:, start_col + 2], errors='coerce')
        rn_data = pd.to_numeric(df.iloc[:, start_col + 3], errors='coerce')        

        num_rows = len(df['Date'])
        # Add the columns in long format to the train and test dataframes
        aux_df_cna =  pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} CNA" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': cna_data.dropna().tolist()
        })
        aux_df_lpn =  pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} LPN" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': lpn_data.dropna().tolist()
        })
        aux_df_rn = pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} RN" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': rn_data.dropna().tolist()
        })
        train_df = pd.concat([train_df, aux_df_cna[:split_idx], aux_df_lpn[:split_idx], aux_df_rn[:split_idx]], ignore_index=True)
        test_df = pd.concat([test_df, aux_df_cna[split_idx:], aux_df_lpn[split_idx:], aux_df_rn[split_idx:]], ignore_index=True)


In [19]:
Y_df = pd.concat([train_df, test_df], ignore_index=True)

y_true = Y_df['y'].to_numpy()
y_pred = np.array([])
lower_bound = np.array([])
upper_bound = np.array([])
for k,v in nh_data.items():
    y_pred = np.concatenate((y_pred, np.repeat(v['CNA Point Prediction'].values, 5)))
    y_pred = np.concatenate((y_pred, np.repeat(v['LPN Point Prediction'].values, 5)))
    y_pred = np.concatenate((y_pred, np.repeat(v['RN Point Prediction'].values, 5)))
    lower_bound = np.concatenate((lower_bound, np.repeat(v['CNA Lower Bound'].values, 5)))
    lower_bound = np.concatenate((lower_bound, np.repeat(v['LPN Lower Bound'].values, 5)))
    lower_bound = np.concatenate((lower_bound, np.repeat(v['RN Lower Bound'].values, 5)))
    upper_bound = np.concatenate((upper_bound, np.repeat(v['CNA Upper Bound'].values, 5)))
    upper_bound = np.concatenate((upper_bound, np.repeat(v['LPN Upper Bound'].values, 5)))
    upper_bound = np.concatenate((upper_bound, np.repeat(v['RN Upper Bound'].values, 5)))
print(y_true)
print(y_pred)
print(lower_bound)
print(upper_bound)

[87.72 82.48 78.47 ... 12.75  8.    8.  ]
[131.3062286376953 131.3062286376953 131.3062286376953 ...
 29.35039520263672 29.35039520263672 29.35039520263672]
[104.4436492919922 104.4436492919922 104.4436492919922 ...
 19.18384170532227 19.18384170532227 19.18384170532227]
[172.9611663818359 172.9611663818359 172.9611663818359 ...
 41.31904411315918 41.31904411315918 41.31904411315918]


In [20]:
print(len(y_true))
print(len(y_pred))
print(len(lower_bound))
print(len(upper_bound))

27300
27300
27300
27300


In [21]:
def calculate_mae(y_true, y_pred):
    """Calculate Mean Absolute Error"""
    return np.mean(np.abs(y_true - y_pred))

def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error with handling for small values"""
    # Use a threshold to avoid division by very small numbers
    threshold = 1.0
    mask = y_true > threshold
    if not np.any(mask):
        return np.nan
    return 100 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def calculate_smape(y_true, y_pred):
    """Calculate Symmetric Mean Absolute Percentage Error"""
    return 100 * np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def calculate_mis(y_true, y_pred, lower_bound, upper_bound):
    """Calculate Mean Interval Score with robust handling of outliers"""
    alpha = 0.05
    n = len(y_true)
    
    # Initialize components of MIS
    coverage_penalty = np.zeros(n)
    width_penalty = np.zeros(n)
    
    # Calculate penalties with outlier-robust handling
    for i in range(n):
        interval_width = upper_bound[i] - lower_bound[i]
        
        if y_true[i] < lower_bound[i]:
            coverage_penalty[i] = 2/alpha * (lower_bound[i] - y_true[i])
        elif y_true[i] > upper_bound[i]:
            coverage_penalty[i] = 2/alpha * (y_true[i] - upper_bound[i])
            
        width_penalty[i] = interval_width
        
    # Use median instead of mean for more robustness
    mis = np.median(width_penalty + coverage_penalty)
    return mis

def evaluate_predictions(y_true, y_pred, lower_bound, upper_bound):
    """Evaluate predictions using all metrics"""
    metrics = {
        'MAE': calculate_mae(y_true, y_pred),
        'MAPE': calculate_mape(y_true, y_pred),
        'SMAPE': calculate_smape(y_true, y_pred),
        'MIS': calculate_mis(y_true, y_pred, lower_bound, upper_bound)
    }
    return metrics
    
evaluate_predictions(y_true, y_pred, lower_bound, upper_bound)

{'MAE': 77.61424030354172,
 'MAPE': 210.82439545127164,
 'SMAPE': 84.6650391639478,
 'MIS': 1390.560962677002}